# Retrieval-Augmented Generation (RAG)

In [1]:
import chromadb
import dotenv
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [2]:
# We populated the RAG with the data from the data/calories.csv file in
# the rag_setup.ipynb notebook

chroma_client = chromadb.PersistentClient("../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")
nutrition_qna_db = chroma_client.get_collection(name="nutrition_qna")


In [3]:
results = nutrition_db.query(query_texts=["banana"], n_results=2)
for i, doc in enumerate(results["documents"][0]):
    print(sorted(results["metadatas"][0][i].items()))
    print(doc)
    print("\n")

/home/vscode/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 104MiB/s] 


[('calories_per_100g', 89.0), ('food_category', 'fruits'), ('food_item', 'banana'), ('keywords', 'banana_fruits'), ('kj_per_100g', 374.0), ('serving_info', '100g')]
Food: Banana
        Category: Fruits
        Nutritional Information:
        - Calories: 89 per 100g
        - Energy: 374 kJ per 100g
        - Serving size reference: 100g

        This is a fruits food item that provides 89 calories per 100 grams.


[('calories_per_100g', 50.0), ('food_category', '(fruit)juices'), ('food_item', 'banana juice'), ('keywords', 'banana_juice_(fruit)juices'), ('kj_per_100g', 210.0), ('serving_info', '100ml')]
Food: Banana Juice
        Category: (Fruit)Juices
        Nutritional Information:
        - Calories: 50 per 100g
        - Energy: 210 kJ per 100g
        - Serving size reference: 100ml

        This is a (fruit)juices food item that provides 50 calories per 100 grams.




In [3]:
@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

@function_tool
def nutrition_qna_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function to ask a question about nutrition.

    Args:
        query: The question to ask
        max_results: Tha maximum number of results to return.

    Returns:
        A string containing the question and answer related to the query.
    """

    results = nutrition_qna_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No information found for: {query}"

    # format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        formatted_results.append(doc)

    return "Related answers to your question:\n" + "\n".join(formatted_results)


Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [ ]:
# calorie_lookup_tool('bananas')

'Nutrition Information:\nBanana (Fruits): 89.0 calories per 100g\nBanana Juice ((Fruit)Juices): 50.0 calories per 100g\nBanana Nut Bread (Pastries,Breads&Rolls): 326.0 calories per 100g'

In [4]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information and nutrition advice.
    You give concise answers.
    If you need to look up calorie information, use the calorie_lookup_tool.
    If you are asked a question about nutrition, always use the nutrition_qna_tool first to see if tehre is an answer in the knowledge database.
    """,
    tools=[calorie_lookup_tool, nutrition_qna_tool]
)

In [8]:
with trace("Nutrition Assistant with Nutrition and Calorie RAG"):
    result = await Runner.run(
        calorie_agent,
        ##"What are the best meal choices for pregnant women and how many calories do they have?"
        ##"What additional nutritional requirements are there for pregnant women?"
        ##"What are the reccommended dietary modifications during pregnancy?"
        "What kinds of food are important for pregnant ladies?"
    )
    print(result.final_output)

Key foods for pregnancy (in general):

- Cereals & millets: whole grains for energy and folate
- Protein: dairy, eggs, lean meats, fish (low-mercury), beans/lentils, tofu
- Fruits & vegetables: aim for a variety; include dark green leafy veggies (folate, iron) and vitamin C–rich produce
- Dairy or fortified alternatives: milk, yogurt, cheese (calcium and protein)
- Healthy fats: fatty fish (low mercury), nuts, seeds, olive oil
- Fiber & fluids: whole grains, fruits, vegetables, water, and other non-caffeinated beverages

Important nutrients to focus on:
- Folate/folic acid: leafy greens, legumes, fortified grains
- Iron: red meat, poultry, fish, beans, fortified cereals (plus vitamin C to help absorption)
- Calcium: dairy or fortified plant milks/yogurt
- Protein: included at each meal
- DHA (omega-3): certain fish; consider prenatal vitamin with DHA if advised
- Fiber: to help with constipation

Foods to avoid or limit:
- High-mercury fish (e.g., some large predator fish)
- Raw or und